# CP-aware generation and simultaneous fit closure

This notebook demonstrates a minimal direct-CP-violation closure test with shared hadronic parameters.

For one amplitude component we define
\[
c_q=(x+q\,\Delta x)+i(y+q\,\Delta y),\qquad q=\pm1.
\]

Thus
\[
c_+=(x+\Delta x)+i(y+\Delta y),\qquad
c_-=(x-\Delta x)+i(y-\Delta y).
\]

The same four fit parameters are shared by the two charge samples. A fixed reference amplitude provides the phase convention and makes the real/imaginary CP differences measurable through interference.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from dataclasses import dataclass

from dalitzplotfitter import (
    CPRealImag, DalitzAmplitude, DecayChannel, DecayModel, Minimizer,
    NonResonant, Parameter, RealImag, enable_x64, weighted_resample,
)
from dalitzplotfitter.likelihood import SimultaneousNLL

enable_x64()


## A simple fixed Dalitz shape

For this first closure test the second amplitude is deliberately generic rather than tied to a particular resonance. The goal is to validate the CP-coefficient and simultaneous-fit machinery in isolation. Once this closes, the same coefficient can be attached to any physical `Resonance` component.


In [ ]:
@dataclass(frozen=True)
class ComplexBand:
    center: float = 1.0
    width: float = 0.28
    phase_slope: float = 1.1

    @property
    def parameters(self):
        return {}

    def __call__(self, data, parameters=None):
        del parameters
        s12 = jnp.asarray(data["s12"])
        envelope = jnp.exp(-0.5*((s12-self.center)/self.width)**2)
        phase = self.phase_slope*(s12-self.center)
        return envelope*jnp.exp(1j*phase)


## Shared CP parameters and the two charge models

The reference coefficient is fixed to \(1+0i\). The second component uses the same `Parameter` objects in both charge models, with only `charge=+1` or `charge=-1` changing.


In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))

x  = Parameter.coefficient("band.x",  0.75, owner="band", step=0.02)
y  = Parameter.coefficient("band.y", -0.25, owner="band", step=0.02)
dx = Parameter.coefficient("band.dx", 0.12, owner="band", step=0.01)
dy = Parameter.coefficient("band.dy", 0.08, owner="band", step=0.01)

cp = CPRealImag(x, y, dx, dy, charge=+1)

def build_model(charge):
    coefficient = cp.for_charge(charge)
    return DecayModel(
        channel,
        [
            NonResonant(RealImag(1.0, 0.0), name="reference"),
            DalitzAmplitude("band", ComplexBand(), coefficient),
        ],
        normalization_resolution=280,
    )

model_plus = build_model(+1)
model_minus = build_model(-1)

truth = {p.name: float(p.value) for p in model_plus.parameters}
print("truth:", truth)
print("c+ =", complex(cp.for_charge(+1).value(truth)))
print("c- =", complex(cp.for_charge(-1).value(truth)))


## Generate independent \(+\) and \(-\) toys

The two samples use identical phase-space kinematics but different complex coefficients. We generate a large proposal pool for each charge and resample according to \(w_{\rm PS}|A_q|^2\).


In [ ]:
N_POOL = 400_000
N_TOY = 40_000

pool_plus = model_plus.generate_phase_space(N_POOL, seed=8121)
pool_minus = model_minus.generate_phase_space(N_POOL, seed=8122)

target_plus = pool_plus.weights * model_plus.intensity(pool_plus.as_dict(), truth)
target_minus = pool_minus.weights * model_minus.intensity(pool_minus.as_dict(), truth)

toy_plus = weighted_resample(
    jax.random.key(8123), pool_plus, target_plus, N_TOY, replace=True
)
toy_minus = weighted_resample(
    jax.random.key(8124), pool_minus, target_minus, N_TOY, replace=True
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
axes[0].hist2d(np.asarray(toy_plus.s12), np.asarray(toy_plus.s13), bins=80)
axes[0].set(title="charge +", xlabel=r"$s_{12}$", ylabel=r"$s_{13}$")
axes[1].hist2d(np.asarray(toy_minus.s12), np.asarray(toy_minus.s13), bins=80)
axes[1].set(title="charge -", xlabel=r"$s_{12}$", ylabel=r"$s_{13}$")
plt.show()


## Cached simultaneous likelihood

Each charge gets its own `PreparedAmplitudeCache`, but both likelihood terms receive the same flat parameter mapping. Since only coefficient parameters float, neither lineshapes nor normalization matrices are recomputed during minimization.


In [ ]:
cache_plus = model_plus.prepare_cache(toy_plus)
cache_minus = model_minus.prepare_cache(toy_minus)

def cached_nll(cache):
    def term(values):
        intensity, normalization = cache.evaluate(values)
        tiny = jnp.finfo(intensity.dtype).tiny
        return -jnp.sum(jnp.log(jnp.maximum(intensity, tiny))) + intensity.shape[0]*jnp.log(normalization)
    return term

objective = SimultaneousNLL((
    cached_nll(cache_plus),
    cached_nll(cache_minus),
))

parameters = model_plus.parameters
fitter = Minimizer(objective, parameters, tolerance=1e-5, verbose=1)


## Randomize once and fit

This is a single-start closure test: the truth parameters generate the toys, the starting point is randomized, and one simultaneous fit is performed.


In [ ]:
start = fitter.random_start(seed=20260830)
print("start:", start)

result = fitter.fit(start_values=start, simplex=False, ncall=10000)
print(result.fmin)


## Compare truth, start, and fitted values


In [ ]:
names = [p.name for p in parameters if not p.fixed]
fit_values = {name: float(result.values[name]) for name in names}
fit_errors = {name: float(result.errors[name]) for name in names}

rows = []
for name in names:
    pull = (fit_values[name] - truth[name]) / fit_errors[name]
    rows.append((name, truth[name], start[name], fit_values[name], fit_errors[name], pull))

print(f"{'parameter':12s} {'truth':>10s} {'start':>10s} {'fit':>10s} {'error':>10s} {'pull':>9s}")
for row in rows:
    print(f"{row[0]:12s} {row[1]:10.4f} {row[2]:10.4f} {row[3]:10.4f} {row[4]:10.4f} {row[5]:9.3f}")


## Reconstructed charge-dependent coefficients


In [ ]:
cplus_truth = complex(cp.for_charge(+1).value(truth))
cminus_truth = complex(cp.for_charge(-1).value(truth))
cplus_fit = complex(cp.for_charge(+1).value(fit_values))
cminus_fit = complex(cp.for_charge(-1).value(fit_values))

print("truth c+:", cplus_truth, " fitted c+:", cplus_fit)
print("truth c-:", cminus_truth, " fitted c-:", cminus_fit)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter([cplus_truth.real, cminus_truth.real], [cplus_truth.imag, cminus_truth.imag],
           marker="x", s=100, label="truth")
ax.scatter([cplus_fit.real, cminus_fit.real], [cplus_fit.imag, cminus_fit.imag],
           marker="o", s=60, label="fit")
for label, z in [("+ truth", cplus_truth), ("- truth", cminus_truth),
                 ("+ fit", cplus_fit), ("- fit", cminus_fit)]:
    ax.annotate(label, (z.real, z.imag))
ax.axhline(0, lw=0.7)
ax.axvline(0, lw=0.7)
ax.set(xlabel="Re(c)", ylabel="Im(c)", title="CP coefficients: truth vs fit")
ax.legend()
plt.show()
